# AVION + SMS Loss — EPIC-KITCHENS-100 MIR

Replicates the 2024 winning approach (~74% nDCG) using:
- **AVION** backbone (CLIP ViT-L pre-trained on Ego4D via LaViLa)
- **SMS Loss** (Symmetric Multi-Similarity Loss with relevancy matrix)

## Prerequisites — do these BEFORE running any cell

### 1. Runtime: GPU → A100
Runtime → Change runtime type → A100

### 2. Add AVION pre-processed videos to your Drive
Open this link and click **"Add shortcut to Drive"** → My Drive:
```
https://drive.google.com/file/d/13J2uC2g2H_DEHrBvgr5Aiu0BgqlCvWqG/view
```
If it is a zip file, also run the extraction cell (Cell 5b). The folder must be named `EK100_320p_15sec_30fps_libx264`.

### 3. Upload annotation files to your Drive
From your local machine, upload these files to `MyDrive/EK100_annotations/`:
```
EK100_MIR/data/MI-MM/dataframes/EPIC_100_retrieval_train.csv
EK100_MIR/data/MI-MM/dataframes/EPIC_100_retrieval_test.csv
EK100_MIR/data/MI-MM/relevancy/caption_relevancy_EPIC_100_retrieval_train.pkl
```
The test relevancy (`caption_relevancy_EPIC_100_retrieval_test.pkl`) is bundled
inside the AVION GDrive archive. If missing, local nDCG metrics will be skipped
but the submission file is still generated.

In [ ]:
# Cell 1 — GPU check
!nvidia-smi
import torch
print(f"PyTorch {torch.__version__}")
print(f"CUDA available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"Device: {torch.cuda.get_device_name(0)}")
    print(f"VRAM: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")

Tue May  5 23:10:51 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  NVIDIA A100-SXM4-80GB          Off |   00000000:00:05.0 Off |                    0 |
| N/A   35C    P0             50W /  400W |       0MiB /  81920MiB |      0%      Default |
|                                         |                        |             Disabled |
+-----------------------------------------+-----

In [ ]:
# Cell 2 — Install dependencies
!apt-get install -qq ffmpeg libavcodec-dev libavformat-dev libavutil-dev libswscale-dev

# Remove flash_attn if it is already installed — any pre-installed version is
# likely compiled for a different torch version and will crash with undefined symbols.
!pip uninstall -q -y flash-attn flash_attn 2>/dev/null; echo "flash_attn removed (or was not present)"

!pip install -q einops kornia timm transformers

# decord: try standard, fall back to eva-decord (maintained fork, same namespace)
import subprocess, sys
r = subprocess.run([sys.executable, "-m", "pip", "install", "-q", "decord"],
                   capture_output=True, text=True)
if r.returncode != 0:
    print("decord failed, falling back to eva-decord...")
    !pip install -q eva-decord
else:
    print("decord installed.")

!pip install -q git+https://github.com/openai/CLIP.git

import importlib
HAS_FLASH_ATTN = importlib.util.find_spec("flash_attn") is not None
print(f"flash_attn available: {HAS_FLASH_ATTN}")   # should be False
print("Done.")

flash_attn removed (or was not present)
decord installed.
  Preparing metadata (setup.py) ... done
flash_attn available: False
Done.


In [ ]:
!pip install ninja

In [ ]:
!pip install torch=='2.4.1+cu121' torchvision=='0.19.1+cu121' torchaudio=='2.4.1+cu121' --index-url https://download.pytorch.org/whl/cu121


Looking in indexes: https://download.pytorch.org/whl/cu121


In [ ]:
!pip install flash-attn

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 8.4/8.4 MB 123.7 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
  Created wheel for flash-attn: filename=flash_attn-2.8.3-cp312-cp312-linux_x86_64.whl size=255985226 sha256=ee1fbb7dc9d4f6e973687e15e245727d39c2b5f5884c74fa30d21bd1841854af
  Stored in directory: /root/.cache/pip/wheels/3d/59/46/f282c12c73dd4bb3c2e3fe199f1a0d0f8cec06df0cccfeee27
Successfully built flash-attn


In [ ]:
import importlib
HAS_FLASH_ATTN = importlib.util.find_spec("flash_attn") is not None
print(f"flash_attn available: {HAS_FLASH_ATTN}")   # should be False
print("Done.")

flash_attn available: True
Done.


In [ ]:
# Cell 3 — Mount Google Drive
from google.colab import drive
drive.mount('/content/drive', force_remount=True)

Mounted at /content/drive


In [ ]:
# Cell 4 — Configure paths
import os

GDRIVE = "/content/drive/MyDrive"

# Path to the AVION pre-processed EK-100 videos (320p, 15-sec chunks)
# Must contain participant folders: P01/, P02/, ..., P37/
DATA_ROOT = f"{GDRIVE}/EK100_320p_15sec_30fps_libx264"

# Annotation files — upload from local EK100_MIR/data/ before running
ANNOT_DIR = f"{GDRIVE}/EK100_annotations"

# Experiment output (saved to Drive so it survives session disconnect)
EXP_DIR = f"{GDRIVE}/experiments/sms_vitl"

# AVION ViT-L pretrain checkpoint (downloaded in Cell 6)
PRETRAIN_CKPT = f"{GDRIVE}/checkpoints/avion_pretrain_lavila_vitl_best.pt"

os.makedirs(ANNOT_DIR, exist_ok=True)
os.makedirs(EXP_DIR, exist_ok=True)
os.makedirs(os.path.dirname(PRETRAIN_CKPT), exist_ok=True)

print(f"DATA_ROOT : {DATA_ROOT}")
print(f"  exists  : {os.path.isdir(DATA_ROOT)}")
print(f"ANNOT_DIR : {ANNOT_DIR}")
print(f"EXP_DIR   : {EXP_DIR}")

DATA_ROOT : /content/drive/MyDrive/EK100_320p_15sec_30fps_libx264
  exists  : True
ANNOT_DIR : /content/drive/MyDrive/EK100_annotations
EXP_DIR   : /content/drive/MyDrive/experiments/sms_vitl


In [ ]:
# Cell 5a — Check video data
# The AVION GDrive link might be a folder shortcut OR a zip file.
# After adding the shortcut, check whether it's already a directory:
if os.path.isdir(DATA_ROOT):
    participants = [d for d in os.listdir(DATA_ROOT) if d.startswith('P')]
    print(f"Found {len(participants)} participant folders: {sorted(participants)[:5]}...")
else:
    print("DATA_ROOT not found as a directory.")
    # Check if the shortcut landed as a zip:
    zip_path = f"{GDRIVE}/EK100_320p_15sec_30fps_libx264.zip"
    if os.path.isfile(zip_path):
        print(f"Found zip at {zip_path}. Run Cell 5b to extract.")
    else:
        print("Neither folder nor zip found. Please add the GDrive shortcut first.")
        print("Link: https://drive.google.com/file/d/13J2uC2g2H_DEHrBvgr5Aiu0BgqlCvWqG/view")

Found 37 participant folders: ['P01', 'P02', 'P03', 'P04', 'P05']...


In [ ]:
# Cell 6 — Download AVION LaViLa ViT-L pretrain checkpoint (~1.3 GB)
# Source: https://github.com/zhaoyue-zephyrus/AVION/blob/main/scripts/download_checkpoints.sh
if not os.path.isfile(PRETRAIN_CKPT):
    print("Downloading AVION ViT-L pretrain checkpoint...")
    !wget --show-progress -O "$PRETRAIN_CKPT" \
        "https://utexas.box.com/shared/static/1iatmrs7ufdeooce09a61t1n6wsouf4l.pt"
else:
    print(f"Checkpoint already present ({os.path.getsize(PRETRAIN_CKPT)/1e9:.2f} GB).")

Checkpoint already present (5.12 GB).


In [ ]:
# Cell 7 — Verify annotation files
# These should have been uploaded from local EK100_MIR/data/ to ANNOT_DIR on Drive.
# If any are missing, this cell downloads them from EPIC-KITCHENS GitHub.
import subprocess

TRAIN_CSV = f"{ANNOT_DIR}/EPIC_100_retrieval_train.csv"
TEST_CSV  = f"{ANNOT_DIR}/EPIC_100_retrieval_test.csv"
TRAIN_REL = f"{ANNOT_DIR}/caption_relevancy_EPIC_100_retrieval_train.pkl"
# Test relevancy is bundled inside the AVION GDrive archive under:
#   epic-kitchens-100-annotations/retrieval_annotations/relevancy/
# Try that path first, then fall back to ANNOT_DIR.
TEST_REL_AVION = (
    f"{DATA_ROOT}/epic-kitchens-100-annotations/"
    "retrieval_annotations/relevancy/"
    "caption_relevancy_EPIC_100_retrieval_test.pkl"
)
TEST_REL = TEST_REL_AVION if os.path.isfile(TEST_REL_AVION) else f"{ANNOT_DIR}/caption_relevancy_EPIC_100_retrieval_test.pkl"

BASE = "https://raw.githubusercontent.com/epic-kitchens/epic-kitchens-100-annotations/master/retrieval_annotations"

for path, url in [
    (TRAIN_CSV, f"{BASE}/EPIC_100_retrieval_train.csv"),
    (TEST_CSV,  f"{BASE}/EPIC_100_retrieval_test.csv"),
]:
    if not os.path.isfile(path):
        print(f"Downloading {os.path.basename(path)} ...")
        subprocess.run(["wget", "-q", "-O", path, url], check=True)

# Train relevancy — GitHub raw works for pkl
if not os.path.isfile(TRAIN_REL):
    url = f"{BASE}/relevancy/caption_relevancy_EPIC_100_retrieval_train.pkl"
    print("Downloading train relevancy ...")
    subprocess.run(["wget", "-q", "-O", TRAIN_REL, url], check=True)

print(f"TRAIN_CSV  : {os.path.isfile(TRAIN_CSV)}")
print(f"TEST_CSV   : {os.path.isfile(TEST_CSV)}")
print(f"TRAIN_REL  : {os.path.isfile(TRAIN_REL)}")
print(f"TEST_REL   : {os.path.isfile(TEST_REL)}  (path: {TEST_REL})")

TRAIN_CSV  : True
TEST_CSV   : True
TRAIN_REL  : True
TEST_REL   : True  (path: /content/drive/MyDrive/EK100_annotations/caption_relevancy_EPIC_100_retrieval_test.pkl)


In [ ]:
# Cell 8 — Clone SMS-Loss repo
import os
os.chdir('/content')
if not os.path.isdir('/content/SMS-Loss'):
    !git clone https://github.com/xqwang14/SMS-Loss.git
os.chdir('/content/SMS-Loss')
print("Working directory:", os.getcwd())
!ls scripts/

Cloning into 'SMS-Loss'...
remote: Enumerating objects: 105, done.
remote: Counting objects: 100% (105/105), done.
remote: Compressing objects: 100% (97/97), done.
remote: Total 105 (delta 27), reused 9 (delta 4), pack-reused 0 (from 0)
Receiving objects: 100% (105/105), 1.45 MiB | 19.58 MiB/s, done.
Resolving deltas: 100% (27/27), done.
Working directory: /content/SMS-Loss
ammplus_finetune.py  dirtrain.py  ensemble.py  test_mir.py


In [ ]:
HAS_FLASH_ATTN

True

In [ ]:
pip install open_clip_torch


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.5/1.5 MB 79.5 MB/s eta 0:00:00


In [ ]:
import subprocess, os
ANNOT_DIR = "/content/drive/MyDrive/EK100_annotations"
BASE = "https://raw.githubusercontent.com/epic-kitchens/epic-kitchens-100-annotations/master/retrieval_annotations"
for fname in ["EPIC_100_retrieval_train_sentence.csv", "EPIC_100_retrieval_test_sentence.csv"]:
    path = f"{ANNOT_DIR}/{fname}"
    if not os.path.isfile(path):
        subprocess.run(["wget", "-q", "-O", path, f"{BASE}/{fname}"], check=True)
        print(f"Downloaded {fname}")
    else:
        print(f"Already present: {fname}")

Already present: EPIC_100_retrieval_train_sentence.csv
Already present: EPIC_100_retrieval_test_sentence.csv


In [ ]:
# Patch ammplus_finetune.py — add images/texts unpacking in the gradient accumulation branch
path = '/content/SMS-Loss/scripts/ammplus_finetune.py'
with open(path, 'r') as f:
    src = f.read()

old = (
    '        else:\n'
    '            # First, cache the features without any gradient tracking.\n'
    '            with torch.no_grad():\n'
)
new = (
    '        else:\n'
    '            images, texts = inputs[0], inputs[1]\n'
    '            # First, cache the features without any gradient tracking.\n'
    '            with torch.no_grad():\n'
)

if old in src:
    src = src.replace(old, new)
    with open(path, 'w') as f:
        f.write(src)
    print("Patched: images/texts unpacking added to gradient accumulation branch.")
else:
    print("Pattern not found — check indentation or already patched:")
    for i, line in enumerate(src.splitlines(), 1):
        if 'First, cache the features' in line:
            print(f"  line {i}: {repr(line)}")

Patched: images/texts unpacking added to gradient accumulation branch.


In [ ]:
path = '/content/SMS-Loss/scripts/ammplus_finetune.py'
with open(path, 'r') as f:
    src = f.read()

# args.accum_freq is referenced but never defined in argparse — it's the same as update_freq
count = src.count('args.accum_freq')
src = src.replace('args.accum_freq', 'args.update_freq')

with open(path, 'w') as f:
    f.write(src)
print(f"Replaced {count} occurrence(s) of args.accum_freq → args.update_freq")

Replaced 1 occurrence(s) of args.accum_freq → args.update_freq


In [ ]:
path = '/content/SMS-Loss/avion/data/clip_dataset.py'
with open(path, 'r') as f:
    src = f.read()

# Patch __getitem__ to retry with a random sample if get_raw_item() returns None
old = '''    def __getitem__(self, i):
        frames, caption, pos = self.get_raw_item('''

new = '''    def __getitem__(self, i):
        import random as _random
        result = None
        _tries = 0
        while result is None and _tries < 20:
            result = self.get_raw_item('''

if old in src:
    src = src.replace(old, new)
    # Now find the closing of the get_raw_item call and add retry logic
    old2 = '''        )

        if isinstance(caption, tuple):'''
    new2 = '''        )
            _tries += 1
            if result is None:
                i = _random.randint(0, len(self) - 1)
        if result is None:
            raise RuntimeError(f"Could not load a valid sample after 20 retries")
        frames, caption, pos = result

        if isinstance(caption, tuple):'''
    src = src.replace(old2, new2, 1)  # only replace first occurrence
    with open(path, 'w') as f:
        f.write(src)
    print("Patched clip_dataset.py — None returns from get_raw_item now retry with a random index.")
else:
    print("Pattern not found. Check indentation:")
    for i, l in enumerate(src.splitlines(), 1):
        if '__getitem__' in l:
            print(f"  line {i}: {repr(l)}")

Patched clip_dataset.py — None returns from get_raw_item now retry with a random index.


In [ ]:
# ── Fix __getitem__ scope bug in clip_dataset.py ─────────────────────────────
import re

path = '/content/SMS-Loss/avion/data/clip_dataset.py'
with open(path) as f:
    src = f.read()

# 1. Locate __getitem__ method boundaries
gi_s = src.index('    def __getitem__(self, i):\n')
nxt = re.search(r'\n    def [a-z_]', src[gi_s + 30:])
gi_e = gi_s + 30 + nxt.start() + 1
old = src[gi_s:gi_e]

# 2. Extract self.get_raw_item(...) using paren-depth counter
call_s = old.index('self.get_raw_item(')
pos = call_s + len('self.get_raw_item(')
depth = 1
while depth:
    c = old[pos]; pos += 1
    if c == '(':   depth += 1
    elif c == ')': depth -= 1
raw_call = old[call_s:pos]          # "self.get_raw_item(i, ..., )"

# 3. Re-indent continuation lines to 16 spaces (inside while loop)
lines = raw_call.split('\n')
reindented = lines[0]               # "self.get_raw_item("
for ln in lines[1:]:
    s = ln.strip()
    reindented += '\n' + ('                ' + s if s else '')

# 4. Post-unpacking code (anchor on "if isinstance(caption, tuple):")
post_s = old.index('        if isinstance(caption, tuple):')
post   = old[post_s:]

# 5. Build corrected method — frames,caption,pos = result is AFTER the loop
fixed = (
    '    def __getitem__(self, i):\n'
    '        import random as _random\n'
    '        result = None\n'
    '        _tries = 0\n'
    '        while result is None and _tries < 20:\n'
    '            result = ' + reindented + '\n'
    '            _tries += 1\n'
    '            if result is None:\n'
    '                i = _random.randint(0, len(self) - 1)\n'
    '        if result is None:\n'
    '            raise RuntimeError("No valid sample after 20 retries")\n'
    '        frames, caption, pos = result\n'
    + post
)

# 6. Write back and preview
with open(path, 'w') as f:
    f.write(src[:gi_s] + fixed + src[gi_e:])

with open(path) as f:
    new = f.read()
gi2 = new.index('    def __getitem__(self, i):\n')
nxt2 = re.search(r'\n    def [a-z_]', new[gi2 + 30:])
end = (gi2 + 30 + nxt2.start() + 1) if nxt2 else (gi2 + 1500)
print(new[gi2:end][:900])
print('\n✓ patch applied')

    def __getitem__(self, i):
        import random as _random
        result = None
        _tries = 0
        while result is None and _tries < 20:
            result = self.get_raw_item(
                i, is_training=self.is_training,
                chunk_len=self.chunk_len,
                clip_length=self.clip_length,
                clip_stride=self.clip_stride,
                threads=self.threads,
                fast_rrc=self.fast_rrc,
                rrc_params=self.rrc_params,
                fast_rcc=self.fast_rcc,
                rcc_params=self.rcc_params,
                )
            _tries += 1
            if result is None:
                i = _random.randint(0, len(self) - 1)
        if result is None:
            raise RuntimeError("No valid sample after 20 retries")
        frames, caption, pos = result
        if isinstance(caption, tuple):
            caption, re

✓ patch applied


In [ ]:
# ── Patch ammplus_finetune.py: checkpoint every epoch + no validation ─────────
import re

path = '/content/SMS-Loss/scripts/ammplus_finetune.py'
with open(path) as f:
    src = f.read()
lines = src.split('\n')

# ── Diagnose: show key lines so we can verify patches hit the right spots ─────
print("=== Key lines ===")
for i, line in enumerate(lines):
    if any(k in line for k in ['validate_mir', 'eval_freq', 'save_on_master',
                                'save_checkpoint', 'checkpoint_path', 'output_dir']):
        print(f"{i+1:4d}│ {line}")

=== Key lines ===
 279│         latest = os.path.join(args.output_dir, 'checkpoint.pt')
 374│         val_stats = validate_mir(val_loader, val_transform_gpu, model, criterion, args)
 376│             with open(os.path.join(args.output_dir, 'eval_log.txt'), 'a') as f:
 394│         if (epoch + 1) % args.eval_freq != 0:
 397│         val_stats = validate_mir(val_loader, val_transform_gpu, model, criterion, args)
 407│         dist_utils.save_on_master({
 414│             }, is_best, args.output_dir)
 421│             with open(os.path.join(args.output_dir, 'log.txt'), 'a') as f:
 554│ def validate_mir(val_loader, transform_gpu, model, criterion, args):
 648│     os.makedirs(args.output_dir, exist_ok=True)


In [ ]:
# ── Patch ammplus_finetune.py: save checkpoint every epoch, no validation ─────
path = '/content/SMS-Loss/scripts/ammplus_finetune.py'
with open(path) as f:
    lines = f.readlines()

# ── Show context so we can confirm the patch targets ─────────────────────────
print("=== Lines 368–420 ===")
for i, line in enumerate(lines[367:420], start=368):
    print(f"{i:4d}│ {line}", end='')

# ── Helpers ───────────────────────────────────────────────────────────────────
def find_line(lines, pattern, start=0):
    for i in range(start, len(lines)):
        if pattern in lines[i]:
            return i
    return -1

def indent_of(line):
    return len(line) - len(line.lstrip())

# ── Find key line indices (0-based) ──────────────────────────────────────────
mir1  = find_line(lines, 'val_stats = validate_mir')
cond  = find_line(lines, 'eval_freq != 0')
mir2  = find_line(lines, 'val_stats = validate_mir', mir1 + 1)
save  = find_line(lines, 'save_on_master')
assert all(x >= 0 for x in [mir1, cond, mir2, save]), \
    f"Pattern not found: mir1={mir1} cond={cond} mir2={mir2} save={save}"
print(f"\nmir1={mir1+1}  cond={cond+1}  mir2={mir2+1}  save={save+1}")

# ── Patch 1: Neutralise BOTH validate_mir calls ───────────────────────────────
for idx in [mir1, mir2]:
    ind = ' ' * indent_of(lines[idx])
    lines[idx] = (
        f"{ind}val_stats = {{'nDCG': 0.0, 'mAP': 0.0, 'avg_nDCG': 0.0, 'avg_mAP': 0.0}}"
        f"  # validation disabled\n"
        f"{ind}# ORIG: {lines[idx].lstrip()}"
    )

# ── Patch 2: Remove the eval_freq skip-block (replaces "if != 0: continue") ──
# Walk the if-body (lines more indented than the condition)
cond_ind = indent_of(lines[cond])
body_end = cond + 1
while body_end < len(lines):
    ln = lines[body_end]
    if ln.strip() and indent_of(ln) <= cond_ind:
        break
    body_end += 1
print(f"\neval_freq block body (lines {cond+2}–{body_end}):")
print(''.join(lines[cond+1:body_end]))
# Delete the entire if-block (condition + body) — checkpoint save below handles it
del lines[cond:body_end]

# ── Patch 3: Add numbered checkpoint save after save_on_master ───────────────
# Re-locate save_on_master after deletion
save = find_line(lines, 'save_on_master')
# Find end of the multi-line call
depth, save_end = 0, save
for i in range(save, min(save + 15, len(lines))):
    for c in lines[i]:
        if c == '(': depth += 1
        elif c == ')': depth -= 1
    if depth <= 0 and i >= save:
        save_end = i
        break
ind = ' ' * indent_of(lines[save])
numbered = [
    f"{ind}# ── per-epoch numbered checkpoint ──────────────────────────────\n",
    f"{ind}if dist_utils.is_main_process():\n",
    f"{ind}    import torch as _t\n",
    f"{ind}    _sd = model.module.state_dict() if hasattr(model, 'module') else model.state_dict()\n",
    f"{ind}    _t.save({{'epoch': epoch + 1, 'state_dict': _sd,\n",
    f"{ind}             'optimizer': optimizer.state_dict(),\n",
    f"{ind}             'scaler': scaler.state_dict()}},\n",
    f"{ind}            os.path.join(args.output_dir, f'checkpoint_{{epoch:04d}}.pt'))\n",
    f"{ind}    print(f'=> saved checkpoint_{{epoch:04d}}.pt')\n",
]
lines[save_end + 1:save_end + 1] = numbered

# ── Write back ────────────────────────────────────────────────────────────────
with open(path, 'w') as f:
    f.writelines(lines)

print("\n✓ Done. Key lines now:")
with open(path) as f:
    nl = f.readlines()
for i, line in enumerate(nl):
    if any(k in line for k in ['validate_mir', 'eval_freq', 'save_on_master',
                                'checkpoint_', 'ORIG:', 'per-epoch']):
        print(f"{i+1:4d}│ {line}", end='')

=== Lines 368–420 ===
 368│         val_dataset, batch_size=args.batch_size, shuffle=False,
 369│         num_workers=args.workers, pin_memory=False, sampler=val_sampler, drop_last=False
 370│     )
 371│     print('len(val_loader) = {}'.format(len(val_loader)))
 372│ 
 373│     if args.evaluate:
 374│         val_stats = validate_mir(val_loader, val_transform_gpu, model, criterion, args)
 375│         if dist_utils.is_main_process():
 376│             with open(os.path.join(args.output_dir, 'eval_log.txt'), 'a') as f:
 377│                 f.write(json.dumps(val_stats) + '\n')
 378│         return
 379│ 
 380│     lr_schedule = cosine_scheduler(
 381│         args.lr, args.lr_end, args.epochs, len(train_loader) // args.update_freq,
 382│         warmup_epochs=args.warmup_epochs, start_warmup_value=args.lr_start
 383│     )
 384│ 
 385│     print(args)
 386│ 
 387│     print("=> beginning training")
 388│     best_acc1 = 0.
 389│     for epoch in range(args.start_epoch, args.epochs):
 

In [ ]:
path = '/content/SMS-Loss/scripts/ammplus_finetune.py'
with open(path) as f:
    src = f.read()

# Replace both occurrences of the dummy val_stats with a complete set of keys
old = "val_stats = {'nDCG': 0.0, 'mAP': 0.0, 'avg_nDCG': 0.0, 'avg_mAP': 0.0}  # validation disabled"
new = ("val_stats = {'nDCG': 0.0, 'mAP': 0.0, 'avg_nDCG': 0.0, 'avg_mAP': 0.0, "
       "'avg_map': 0.0, 'vis_map': 0.0, 'txt_map': 0.0, "
       "'vis_ndcg': 0.0, 'txt_ndcg': 0.0}  # validation disabled")

count = src.count(old)
src = src.replace(old, new)
with open(path, 'w') as f:
    f.write(src)
print(f"Fixed {count} occurrence(s)")

Fixed 2 occurrence(s)


In [ ]:
# Cell 9 — Fine-tune AVION ViT-L with SMS Loss  (single A100, ~10-14 h for 50 epochs)
import os, importlib
os.chdir('/content/SMS-Loss')

GDRIVE    = "/content/drive/MyDrive"
ANNOT_DIR = f"{GDRIVE}/EK100_annotations"
DATA_ROOT = f"{GDRIVE}/EK100_320p_15sec_30fps_libx264"
EXP_DIR   = f"{GDRIVE}/experiments/sms_vitl"
PRETRAIN_CKPT = f"{GDRIVE}/checkpoints/avion_pretrain_lavila_vitl_best.pt"

TRAIN_CSV = f"{ANNOT_DIR}/EPIC_100_retrieval_train.csv"
TEST_CSV  = f"{ANNOT_DIR}/EPIC_100_retrieval_test.csv"
TRAIN_REL = f"{ANNOT_DIR}/caption_relevancy_EPIC_100_retrieval_train.pkl"
TEST_REL_AVION = (
    f"{DATA_ROOT}/epic-kitchens-100-annotations/"
    "retrieval_annotations/relevancy/"
    "caption_relevancy_EPIC_100_retrieval_test.pkl"
)
TEST_REL = TEST_REL_AVION if os.path.isfile(TEST_REL_AVION) else f"{ANNOT_DIR}/caption_relevancy_EPIC_100_retrieval_test.pkl"

HAS_FLASH_ATTN = importlib.util.find_spec("flash_attn") is not None
flash_flag = "--use-flash-attn" if HAS_FLASH_ATTN else ""
print(f"flash_attn: {'enabled' if HAS_FLASH_ATTN else 'disabled'}")

cmd = f"""\
torchrun --nproc_per_node=1 scripts/ammplus_finetune.py \\
  --root "{DATA_ROOT}" \\
  --train-metadata "{TRAIN_CSV}" \\
  --val-metadata   "{TEST_CSV}" \\
  --relevancy-train "{TRAIN_REL}" \\
  --relevancy-test  "{TEST_REL}" \\
  --pretrain-model  "{PRETRAIN_CKPT}" \\
  --model CLIP_VITL14 \\
  --batch-size 48 \\
  --update-freq 1 \\
  --epochs 10 \\
  --lr 2e-5 \\
  --loss-margin 0.6 \\
  --loss-thres  0.1 \\
  --use-fast-conv1 \\
  {flash_flag} \\
  --grad-checkpointing \\
  --output-dir "{EXP_DIR}"
"""
print(cmd)
!{cmd}

In [ ]:
HAS_FLASH_ATTN = importlib.util.find_spec("flash_attn") is not None
flash_flag = "--use-flash-attn" if HAS_FLASH_ATTN else ""
print(f"flash_attn: {'enabled' if HAS_FLASH_ATTN else 'disabled'}")

flash_attn: enabled


In [ ]:
import re
with open('/content/SMS-Loss/avion/models/model_clip.py') as f:
    src = f.read()

names = [m[0] or m[1] for m in re.findall(r'^def (\w+)|^class (\w+)', src, re.MULTILINE)]
print('\n'.join(names))

VideoClassifier
CLIP
CLIP_VITB16
CLIP_VITL14
CLIP_VITL14_336PX


In [ ]:
# Print lines 155-175 of ammplus_finetune.py to see the model construction call
with open('/content/SMS-Loss/scripts/ammplus_finetune.py') as f:
    lines = f.readlines()
for i, line in enumerate(lines[150:180], start=151):
    print(f"{i}\t{line}", end='')

151	    args.pretrain_path = old_args.pretrain_path
152	
153	    if args.project_embed_dim == 0:
154	        args.project_embed_dim = old_args.project_embed_dim
155	    
156	    print("=> creating model: {}".format(args.model))
157	
158	    model = getattr(model_clip, args.model)(
159	        freeze_temperature=True,
160	        use_grad_checkpointing=args.use_grad_checkpointing,
161	        context_length=args.context_length,
162	        vocab_size=args.vocab_size,
163	        patch_dropout=args.patch_dropout,
164	        num_frames=args.clip_length,
165	        drop_path_rate=args.drop_path_rate,
166	        use_fast_conv1=args.use_fast_conv1,
167	        use_flash_attn=args.use_flash_attn,
168	        use_quick_gelu=True,
169	        project_embed_dim=args.project_embed_dim,
170	        pretrain_zoo=args.pretrain_zoo,
171	        pretrain_path=args.pretrain_path,
172	    )
173	    model.logit_scale.requires_grad = False
174	    print('=> inflating PE in models due to different frame

In [ ]:
with open('/content/SMS-Loss/avion/models/model_clip.py') as f:
    lines = f.readlines()
for i, line in enumerate(lines[205:235], start=206):
    print(f"{i}\t{line}", end='')

206	        print("missing_keys: ", missing_keys)
207	        print("unexpected_keys: ", unexpected_keys)
208	    elif pretrain_zoo == "open_clip":
209	        assert pretrain_path is not None
210	        state_dict = torch.load(pretrain_path)
211	        print("=> loading open_clip model")
212	        remapped_state_dict = remap_keys_from_open_clip_to_vit(state_dict, use_flash_attn=use_flash_attn)
213	        missing_keys, unexpected_keys = model.load_state_dict(remapped_state_dict, strict=False)
214	        print("missing_keys: ", missing_keys)
215	        print("unexpected_keys: ", unexpected_keys)
216	    else:
217	        raise NotImplementedError
218	    return model
219	
220	
221	def CLIP_VITL14_336PX(
222	    freeze_temperature=False,
223	    use_grad_checkpointing=False,
224	    use_bidirectional_lm=False,
225	    context_length=77,
226	    vocab_size=49408,
227	    patch_dropout=0.,
228	    drop_path_rate=0.,
229	    num_frames=1,
230	    use_fast_conv1=False,
231	    use_fla

In [ ]:
with open('/content/SMS-Loss/scripts/ammplus_finetune.py') as f:
    lines = f.readlines()
for i, line in enumerate(lines[135:158], start=136):
    print(f"{i}\t{line}", end='')

136	    if args.pretrain_model:
137	        ckpt_path = args.pretrain_model
138	    else:
139	        raise Exception('no checkpoint found, add it by `--pretrain-model ${CHECKPOINT_PATH}`')
140	    ckpt = torch.load(ckpt_path, map_location='cpu')
141	    state_dict = OrderedDict()
142	    for k, v in ckpt['state_dict'].items():
143	        state_dict[k.replace('module.', '')] = v
144	
145	    old_args = ckpt['args']
146	    #adjust the settings from old_args to args
147	    args.model = old_args.model
148	    args.context_length = old_args.context_length
149	    args.vocab_size = old_args.vocab_size
150	    args.pretrain_zoo = old_args.pretrain_zoo
151	    args.pretrain_path = old_args.pretrain_path
152	
153	    if args.project_embed_dim == 0:
154	        args.project_embed_dim = old_args.project_embed_dim
155	    
156	    print("=> creating model: {}".format(args.model))
157	
158	    model = getattr(model_clip, args.model)(


In [ ]:
import re
with open('/content/SMS-Loss/scripts/ammplus_finetune.py') as f:
    src = f.read()
matches = re.findall(r'.{0,50}pretrain.zoo.{0,100}', src)
for m in matches:
    print(m)

    parser.add_argument('--pretrain-zoo', default=None, type=str, help='path of pretrained model')
    args.pretrain_zoo = old_args.pretrain_zoo
        pretrain_zoo=args.pretrain_zoo,


In [ ]:
path = '/content/SMS-Loss/scripts/ammplus_finetune.py'
with open(path) as f:
    src = f.read()

old = (
    "    old_args = ckpt['args']\n"
)
new = (
    "    old_args = ckpt.get('args', args)\n"
)

assert old in src, "Pattern not found — print line 145 of ammplus_finetune.py and share"
src = src.replace(old, new)
with open(path, 'w') as f:
    f.write(src)
print("✓ Fixed: old_args falls back to current args if not in checkpoint")

✓ Fixed: old_args falls back to current args if not in checkpoint


In [ ]:
path = '/content/SMS-Loss/scripts/ammplus_finetune.py'

# Patch A — safe fallback for all old_args fields when checkpoint has no args
with open(path) as f:
    src = f.read()

old = (
    "    old_args = ckpt.get('args', args)\n"
    "    #adjust the settings from old_args to args\n"
    "    args.model = old_args.model\n"
    "    args.context_length = old_args.context_length\n"
    "    args.vocab_size = old_args.vocab_size\n"
    "    args.pretrain_zoo = old_args.pretrain_zoo\n"
    "    args.pretrain_path = old_args.pretrain_path\n"
)
new = (
    "    old_args = ckpt.get('args', None)\n"
    "    #adjust the settings from old_args to args\n"
    "    if old_args is not None:\n"
    "        args.model = old_args.model\n"
    "        args.context_length = old_args.context_length\n"
    "        args.vocab_size = old_args.vocab_size\n"
    "        args.pretrain_zoo = old_args.pretrain_zoo\n"
    "        args.pretrain_path = old_args.pretrain_path\n"
    "    # if no args in checkpoint, keep current args but ensure pretrain_zoo is valid\n"
    "    if not hasattr(args, 'pretrain_zoo') or args.pretrain_zoo is None:\n"
    "        args.pretrain_zoo = 'openai'\n"
    "    if not hasattr(args, 'context_length'):\n"
    "        args.context_length = 77\n"
    "    if not hasattr(args, 'vocab_size'):\n"
    "        args.vocab_size = 49408\n"
    "    if not hasattr(args, 'pretrain_path') or args.pretrain_path is None:\n"
    "        args.pretrain_path = args.pretrain_model\n"
)

assert old in src, "Patch A pattern not found — share lines 144-154"
src = src.replace(old, new)
with open(path, 'w') as f:
    f.write(src)
print("✓ Patch A applied: safe fallbacks for missing checkpoint args")

✓ Patch A applied: safe fallbacks for missing checkpoint args


In [ ]:
path = '/content/SMS-Loss/scripts/ammplus_finetune.py'
with open(path) as f:
    src = f.read()

old = (
    "    if args.project_embed_dim == 0:\n"
    "        args.project_embed_dim = old_args.project_embed_dim\n"
)
new = (
    "    if args.project_embed_dim == 0:\n"
    "        args.project_embed_dim = old_args.project_embed_dim if old_args is not None else 256\n"
)

assert old in src, "Pattern not found — share lines 160-168 of ammplus_finetune.py"
src = src.replace(old, new)
with open(path, 'w') as f:
    f.write(src)
print("✓ Fixed project_embed_dim fallback")

✓ Fixed project_embed_dim fallback


In [ ]:
path = '/content/SMS-Loss/scripts/ammplus_finetune.py'
with open(path) as f:
    src = f.read()

old = (
    "    if checkpoint['args'].clip_length != args.clip_length:\n"
)
new = (
    "    _ckpt_args = checkpoint.get('args', None)\n"
    "    if _ckpt_args is not None and _ckpt_args.clip_length != args.clip_length:\n"
)

assert old in src, "Pattern not found — share lines 268-285 of ammplus_finetune.py"
src = src.replace(old, new)
with open(path, 'w') as f:
    f.write(src)
print("✓ Fixed resume checkpoint['args'] access")

✓ Fixed resume checkpoint['args'] access


In [ ]:
path = '/content/SMS-Loss/scripts/ammplus_finetune.py'
with open(path) as f:
    lines = f.readlines()
for i, line in enumerate(lines[265:290], start=266):
    print(f"{i}\t{line}", end='')

266	    # optionally resume from a checkpoint (takes precedence over autoresume)
267	    if args.resume:
268	        if os.path.isfile(args.resume):
269	            print("=> loading resume checkpoint '{}'".format(args.resume))
270	            checkpoint = torch.load(args.resume, map_location='cpu')
271	            _ckpt_args = checkpoint.get('args', None)
272	    if _ckpt_args is not None and _ckpt_args.clip_length != args.clip_length:
273	                load_temporal_embedding = checkpoint['state_dict']['module.visual.temporal_embedding']
274	                load_temporal_embedding = load_temporal_embedding.unsqueeze(0).permute(0, 2, 1)
275	                new_temporal_embed = F.interpolate(load_temporal_embedding, size=(args.clip_length,), mode='linear').permute(0, 2, 1).squeeze(0)
276	                checkpoint['state_dict']['module.visual.temporal_embedding'] = new_temporal_embed
277	            epoch = checkpoint['epoch'] if 'epoch' in checkpoint else 0
278	            args.star

In [ ]:
path = '/content/SMS-Loss/scripts/ammplus_finetune.py'
with open(path) as f:
    src = f.read()

old = (
    "            checkpoint = torch.load(args.resume, map_location='cpu')\n"
    "            _ckpt_args = checkpoint.get('args', None)\n"
    "    if _ckpt_args is not None and _ckpt_args.clip_length != args.clip_length:\n"
    "                load_temporal_embedding = checkpoint['state_dict']['module.visual.temporal_embedding']\n"
    "                load_temporal_embedding = load_temporal_embedding.unsqueeze(0).permute(0, 2, 1)\n"
    "                new_temporal_embed = F.interpolate(load_temporal_embedding, size=(args.clip_length,), mode='linear').permute(0, 2, 1).squeeze(0)\n"
    "                checkpoint['state_dict']['module.visual.temporal_embedding'] = new_temporal_embed\n"
    "            epoch = checkpoint['epoch'] if 'epoch' in checkpoint else 0\n"
    "            args.start_epoch = epoch\n"
    "            result = model.load_state_dict(checkpoint['state_dict'], strict=False)\n"
    "            print(result)\n"
    "            optimizer.load_state_dict(checkpoint['optimizer']) if 'optimizer' in checkpoint else ()\n"
    "            scaler.load_state_dict(checkpoint['scaler']) if 'scaler' in checkpoint else ()\n"
    "            best_acc1 = checkpoint['best_acc1']\n"
)
new = (
    "            checkpoint = torch.load(args.resume, map_location='cpu')\n"
    "            _ckpt_args = checkpoint.get('args', None)\n"
    "            if _ckpt_args is not None and _ckpt_args.clip_length != args.clip_length:\n"
    "                load_temporal_embedding = checkpoint['state_dict']['module.visual.temporal_embedding']\n"
    "                load_temporal_embedding = load_temporal_embedding.unsqueeze(0).permute(0, 2, 1)\n"
    "                new_temporal_embed = F.interpolate(load_temporal_embedding, size=(args.clip_length,), mode='linear').permute(0, 2, 1).squeeze(0)\n"
    "                checkpoint['state_dict']['module.visual.temporal_embedding'] = new_temporal_embed\n"
    "            epoch = checkpoint['epoch'] if 'epoch' in checkpoint else 0\n"
    "            args.start_epoch = epoch\n"
    "            result = model.load_state_dict(checkpoint['state_dict'], strict=False)\n"
    "            print(result)\n"
    "            optimizer.load_state_dict(checkpoint['optimizer']) if 'optimizer' in checkpoint else ()\n"
    "            scaler.load_state_dict(checkpoint['scaler']) if 'scaler' in checkpoint else ()\n"
    "            best_acc1 = checkpoint.get('best_acc1', 0.0)\n"
)

assert old in src, "Pattern not found — copy and share lines 270-284 exactly"
src = src.replace(old, new)
with open(path, 'w') as f:
    f.write(src)
print("✓ Fixed resume block indentation and best_acc1 KeyError")

✓ Fixed resume block indentation and best_acc1 KeyError


In [ ]:
path = '/content/SMS-Loss/scripts/ammplus_finetune.py'
with open(path) as f:
    src = f.read()

# Fix 1: old_args.context_length at line 304
old = (
    "    tokenizer = partial(tokenize, context_length=old_args.context_length)\n"
)
new = (
    "    tokenizer = partial(tokenize, context_length=args.context_length)\n"
)

assert old in src, "Fix 1 pattern not found — share line 304"
src = src.replace(old, new)
with open(path, 'w') as f:
    f.write(src)
print("✓ Fix 1: tokenizer uses args.context_length instead of old_args")

✓ Fix 1: tokenizer uses args.context_length instead of old_args


In [ ]:
path = '/content/SMS-Loss/scripts/ammplus_finetune.py'
with open(path) as f:
    src = f.read()

# Fix 2: strip 'module.' prefix when loading resume checkpoint state_dict
old = (
    "            result = model.load_state_dict(checkpoint['state_dict'], strict=False)\n"
    "            print(result)\n"
)
new = (
    "            _resume_sd = OrderedDict()\n"
    "            for k, v in checkpoint['state_dict'].items():\n"
    "                _resume_sd[k.replace('module.', '')] = v\n"
    "            result = model.load_state_dict(_resume_sd, strict=False)\n"
    "            print(result)\n"
)

assert old in src, "Fix 2 pattern not found — share lines 279-281"
src = src.replace(old, new)
with open(path, 'w') as f:
    f.write(src)
print("✓ Fix 2: module. prefix stripped from resume state_dict")

✓ Fix 2: module. prefix stripped from resume state_dict


In [ ]:
path = '/content/SMS-Loss/scripts/ammplus_finetune.py'
with open(path) as f:
    src = f.read()

old = (
    "    crop_size = 336 if old_args.model.endswith(\"_336PX\") else 224\n"
)
new = (
    "    crop_size = 336 if args.model.endswith(\"_336PX\") else 224\n"
)

assert old in src, "Pattern not found — share line 315"
src = src.replace(old, new)

# Also blanket-replace any remaining old_args. references to args.
# This is safe because old_args is None and args already has all the correct values
import re
remaining = re.findall(r'old_args\.\w+', src)
if remaining:
    print(f"Additional old_args references found: {set(remaining)}")
    src = src.replace('old_args.', 'args.')

with open(path, 'w') as f:
    f.write(src)
print("✓ All remaining old_args references replaced with args")

Additional old_args references found: {'old_args.model', 'old_args.pretrain_zoo', 'old_args.vocab_size', 'old_args.project_embed_dim', 'old_args.pretrain_path', 'old_args.context_length'}
✓ All remaining old_args references replaced with args


In [ ]:
with open('/content/SMS-Loss/scripts/ammplus_finetune.py') as f:
    src = f.read()

# Check state of the file
print("Has dummy val_stats (old):", "val_stats = {'nDCG': 0.0, 'mAP': 0.0, 'avg_nDCG': 0.0, 'avg_mAP': 0.0}  # validation disabled" in src)
print("Has per-epoch checkpoint:", "checkpoint_{epoch:04d}" in src)
print("Has eval_freq skip:", "eval_freq != 0" in src)

Has dummy val_stats (old): False
Has per-epoch checkpoint: True
Has eval_freq skip: False


In [ ]:
CKPT = f"{EXP_DIR}/checkpoint_round_1.pt"

cmd = f"""\
torchrun --nproc_per_node=1 scripts/ammplus_finetune.py \\
  --root "{DATA_ROOT}" \\
  --train-metadata "{TRAIN_CSV}" \\
  --val-metadata   "{TEST_CSV}" \\
  --relevancy-train "{TRAIN_REL}" \\
  --relevancy-test  "{TEST_REL}" \\
  --pretrain-model  "{CKPT}" \\
  --resume          "{CKPT}" \\
  --pretrain-zoo    openai \\
  --model CLIP_VITL14 \\
  --batch-size 48 \\
  --update-freq 1 \\
  --epochs 50 \\
  --start-epoch 10 \\
  --lr 2e-5 \\
  --loss-margin 0.6 \\
  --loss-thres  0.1 \\
  --use-fast-conv1 \\
  {flash_flag} \\
  --grad-checkpointing \\
  --output-dir "{EXP_DIR}"
"""
print(cmd)
!{cmd}

torchrun --nproc_per_node=1 scripts/ammplus_finetune.py \
  --root "/content/drive/MyDrive/EK100_320p_15sec_30fps_libx264" \
  --train-metadata "/content/drive/MyDrive/EK100_annotations/EPIC_100_retrieval_train.csv" \
  --val-metadata   "/content/drive/MyDrive/EK100_annotations/EPIC_100_retrieval_test.csv" \
  --relevancy-train "/content/drive/MyDrive/EK100_annotations/caption_relevancy_EPIC_100_retrieval_train.pkl" \
  --relevancy-test  "/content/drive/MyDrive/EK100_annotations/caption_relevancy_EPIC_100_retrieval_test.pkl" \
  --pretrain-model  "/content/drive/MyDrive/experiments/sms_vitl/checkpoint_round_1.pt" \
  --resume          "/content/drive/MyDrive/experiments/sms_vitl/checkpoint_round_1.pt" \
  --pretrain-zoo    openai \
  --model CLIP_VITL14 \
  --batch-size 48 \
  --update-freq 1 \
  --epochs 50 \
  --start-epoch 10 \
  --lr 2e-5 \
  --loss-margin 0.6 \
  --loss-thres  0.1 \
  --use-fast-conv1 \
  --use-flash-attn \
  --grad-checkpointing \
  --output-dir "/content/drive/M

In [ ]:



# ── Patch test_mir.py ────────────────────────────────────────────────────────
path = '/content/SMS-Loss/scripts/test_mir.py'
with open(path) as f:
    src = f.read()

# 1. Fix torch.load + handle missing ckpt['args']
old = (
    "    ckpt = torch.load(ckpt_path, map_location='cpu')\n"
    "    state_dict = OrderedDict()\n"
    "    for k, v in ckpt['state_dict'].items():\n"
    "        state_dict[k.replace('module.', '')] = v\n\n"
    "    old_args = ckpt['args']"
)
new = (
    "    # ── serialization compat fix ────────────────────────────────────\n"
    "    import sys, types as _t\n"
    "    if 'torch.utils.serialization' not in sys.modules:\n"
    "        _m = _t.ModuleType('torch.utils.serialization')\n"
    "        class _ST:\n"
    "            _map = {'FloatStorage': torch.FloatStorage,\n"
    "                    'DoubleStorage': torch.DoubleStorage,\n"
    "                    'HalfStorage': torch.HalfStorage,\n"
    "                    'ByteStorage': torch.ByteStorage,\n"
    "                    'CharStorage': torch.CharStorage,\n"
    "                    'ShortStorage': torch.ShortStorage,\n"
    "                    'IntStorage': torch.IntStorage,\n"
    "                    'LongStorage': torch.LongStorage,\n"
    "                    'BFloat16Storage': torch.BFloat16Storage}\n"
    "            def __new__(cls, name): return cls._map.get(name, torch.FloatStorage)\n"
    "        _m.StorageType = _ST\n"
    "        sys.modules['torch.utils.serialization'] = _m\n"
    "        import torch.utils as _tu; setattr(_tu, 'serialization', _m)\n"
    "    # ────────────────────────────────────────────────────────────────\n"
    "    ckpt = torch.load(ckpt_path, map_location='cpu', weights_only=False)\n"
    "    state_dict = OrderedDict()\n"
    "    _sd = ckpt['state_dict'] if 'state_dict' in ckpt else ckpt\n"
    "    for k, v in _sd.items():\n"
    "        state_dict[k.replace('module.', '')] = v\n\n"
    "    old_args = ckpt.get('args', args)\n"
    "    if not hasattr(old_args, 'norm_style'): old_args.norm_style = 'openai'\n"
    "    if not hasattr(old_args, 'model'):      old_args.model = args.model"
)
src = src.replace(old, new)
assert old not in src or new in src, "Patch 1 failed to apply"

# 2. Add --project-embed-dim to argparser
old2 = "    parser.add_argument('--pretrain-model', default='', type=str, help='path of pretrained model')"
new2 = ("    parser.add_argument('--project-embed-dim', default=256, type=int)\n" + old2)
src = src.replace(old2, new2)

# 3. Make relevancy optional — save submission FIRST, then optionally compute metrics
old3 = (
    "    rel_matrix = pd.read_pickle(args.relevancy_path)\n"
    "    vis_map, txt_map, avg_map = get_mAP(similarity_matrix, rel_matrix)\n"
    "    print('mAP: V->T: {:.3f} T->V: {:.3f} AVG: {:.3f}'.format(vis_map, txt_map, avg_map))\n"
    "    vis_nDCG, txt_nDCG, avg_nDCG = get_nDCG(similarity_matrix, rel_matrix)\n"
    "    print('nDCG: V->T: {:.3f} T->V: {:.3f} AVG: {:.3f}'.format(vis_nDCG, txt_nDCG, avg_nDCG))\n\n"
    "    create_and_save_dict(similarity_matrix, text_id, video_id)"
)
new3 = (
    "    import os as _os\n"
    "    _sub = _os.path.join(args.output_dir, 'submission.pkl')\n"
    "    create_and_save_dict(similarity_matrix, text_id, video_id, filename=_sub)\n"
    "    vis_map = txt_map = avg_map = vis_nDCG = txt_nDCG = avg_nDCG = 0.0\n"
    "    _rel = getattr(args, 'relevancy_path', '')\n"
    "    if _rel and _os.path.isfile(_rel):\n"
    "        rel_matrix = pd.read_pickle(_rel)\n"
    "        vis_map, txt_map, avg_map = get_mAP(similarity_matrix, rel_matrix)\n"
    "        print('mAP: V->T: {:.3f} T->V: {:.3f} AVG: {:.3f}'.format(vis_map, txt_map, avg_map))\n"
    "        vis_nDCG, txt_nDCG, avg_nDCG = get_nDCG(similarity_matrix, rel_matrix)\n"
    "        print('nDCG: V->T: {:.3f} T->V: {:.3f} AVG: {:.3f}'.format(vis_nDCG, txt_nDCG, avg_nDCG))\n"
    "    else:\n"
    "        print(f'Submission saved to {_sub} (no relevancy file — skipping metrics)')"
)
src = src.replace(old3, new3)

with open(path, 'w') as f:
    f.write(src)
print("✓ test_mir.py patched")

In [ ]:
!pip install reranking

In [ ]:
path = '/content/SMS-Loss/scripts/test_mir.py'
with open(path) as f:
    src = f.read()

src = src.replace(
    'from reranking import re_ranking, naive_rerank',
    '# from reranking import re_ranking, naive_rerank  # disabled\n'
    're_ranking = naive_rerank = None'
)

with open(path, 'w') as f:
    f.write(src)
print("✓ Fixed. Re-run Cell 10.")

In [ ]:
path = '/content/SMS-Loss/scripts/test_mir.py'
with open(path) as f:
    src = f.read()

# Find and remove everything from train_dataset creation to the break,
# keeping only the val_stats call
old = (
    "    train_dataset = VideoCaptionDatasetCLIP(\n"
    "        args.dataset, args.root, args.train_metadata,\n"
    "        transform=train_transform, is_training=True, tokenizer=tokenizer,\n"
    "        clip_length=args.clip_length, clip_stride=args.clip_stride,\n"
    "        chunk_len=args.video_chunk_length,\n"
    "        threads=args.decode_threads,\n"
    "        fast_rrc=args.fused_decode_crop, rrc_params=(crop_size, (0.5, 1.0)),\n"
    "    )\n"
)
new = "    # train_dataset skipped — not needed for inference\n"
src = src.replace(old, new, 1)

# Also skip train_sampler, train_loader, lr_schedule, and the warm-up loop
import re
# Remove train_sampler
src = re.sub(r'    train_sampler = .*?\n', '    # train_sampler skipped\n', src, count=1)
# Remove train_loader + print
src = re.sub(
    r"    train_loader = torch\.utils\.data\.DataLoader\(\s+train_dataset.*?\)\n"
    r"    print\('len\(train_loader\).*?\n",
    "    # train_loader skipped\n",
    src, count=1, flags=re.DOTALL
)
# Remove lr_schedule
src = re.sub(r'    lr_schedule = cosine_scheduler\(.*?\)\n', '    # lr_schedule skipped\n', src, count=1, flags=re.DOTALL)
# Remove the warm-up loop (for data_iter ... break)
src = re.sub(
    r'    for data_iter, inputs in enumerate\(train_loader\):.*?break\n',
    '    # warm-up loop skipped\n',
    src, count=1, flags=re.DOTALL
)

with open(path, 'w') as f:
    f.write(src)
print("✓ Skipped train_dataset. Re-run Cell 10.")

In [ ]:
import sys
sys.path.insert(0, '/content/SMS-Loss')
from avion.data.clip_dataset import VideoCaptionDatasetCLIP
import inspect
print(inspect.signature(VideoCaptionDatasetCLIP.__init__))

In [ ]:
path = '/content/SMS-Loss/scripts/test_mir.py'
with open(path) as f:
    src = f.read()

old = (
    "    val_dataset = VideoCaptionDatasetCLIP(\n"
    "        args.dataset, args.root, args.val_metadata,\n"
    "        transform=val_transform, is_training=False, tokenizer=tokenizer,\n"
    "        clip_length=args.clip_length, clip_stride=args.clip_stride,\n"
    "        chunk_len=args.video_chunk_length,\n"
    "        fast_rcc=args.fused_decode_crop, rcc_params=(crop_size,),\n"
    "    )"
)
new = (
    "    val_dataset = VideoCaptionDatasetCLIP(\n"
    "        args, args.val_metadata,\n"
    "        transform=val_transform, is_training=False, tokenizer=tokenizer,\n"
    "        clip_length=args.clip_length, clip_stride=args.clip_stride,\n"
    "        chunk_len=args.video_chunk_length,\n"
    "        fast_rcc=args.fused_decode_crop, rcc_params=(crop_size,),\n"
    "    )"
)

assert old in src, "Pattern not found — print lines 290-305 of test_mir.py and share"
src = src.replace(old, new)
with open(path, 'w') as f:
    f.write(src)
print("✓ Fixed val_dataset call. Re-run Cell 10.")

In [ ]:
# ── Patch clip_dataset.py line 160 (relevancy_mat load) ──────────────────────
import re

path1 = "/content/SMS-Loss/avion/data/clip_dataset.py"
with open(path1, "r") as f:
    src = f.read()

old = "self.relevancy_mat = pickle.load(open(args.relevancy_test, 'rb'))"
new = (
    "import os as _os\n"
    "        _rel = getattr(args, 'relevancy_test', None) or getattr(args, 'relevancy_path', None)\n"
    "        self.relevancy_mat = pickle.load(open(_rel, 'rb')) if _rel and _os.path.exists(_rel) else None"
)

if old in src:
    src = src.replace(old, new)
    with open(path1, "w") as f:
        f.write(src)
    print("clip_dataset.py patched ✓")
else:
    print("clip_dataset.py: pattern not found — check manually")
    print("Looking for:", repr(old))

# ── Patch test_mir.py: add args.relevancy_test alias before val_dataset ───────
path2 = "/content/SMS-Loss/scripts/test_mir.py"
with open(path2, "r") as f:
    src2 = f.read()

# Anchor on the val_dataset creation line
anchor = "val_dataset = VideoCaptionDatasetCLIP("
alias  = "# alias missing arg\n    if not hasattr(args, 'relevancy_test'):\n        args.relevancy_test = None\n    "

if alias not in src2 and anchor in src2:
    src2 = src2.replace(anchor, alias + anchor)
    with open(path2, "w") as f:
        f.write(src2)
    print("test_mir.py patched ✓")
elif alias in src2:
    print("test_mir.py: already patched ✓")
else:
    print("test_mir.py: anchor not found — check manually")

In [ ]:
# ── Re-patch clip_dataset.py (fix IndentationError from previous patch) ───────
path1 = "/content/SMS-Loss/avion/data/clip_dataset.py"
with open(path1, "r") as f:
    lines = f.readlines()

new_lines = []
i = 0
while i < len(lines):
    line = lines[i]
    stripped = line.rstrip()

    # Remove any broken patch lines we may have introduced
    if ("import os as _os" in stripped and "_rel" not in stripped and
            stripped.strip().startswith("import os as _os")):
        i += 1
        continue
    if "_rel = getattr(args, 'relevancy_test'" in stripped:
        i += 1
        continue
    if "self.relevancy_mat = pickle.load(open(_rel" in stripped:
        i += 1
        continue

    # Replace the original broken line (both old and any half-patched form)
    if ("self.relevancy_mat = pickle.load(open(args.relevancy_test" in stripped or
            "self.relevancy_mat = pickle.load(open(args.relevancy_path" in stripped):
        indent = len(line) - len(line.lstrip())
        pad = " " * indent
        new_lines.append(f"{pad}import os as _os\n")
        new_lines.append(f"{pad}_rel = getattr(args, 'relevancy_test', None) or getattr(args, 'relevancy_path', None)\n")
        new_lines.append(f"{pad}self.relevancy_mat = pickle.load(open(_rel, 'rb')) if _rel and _os.path.exists(_rel) else None\n")
        i += 1
        continue

    new_lines.append(line)
    i += 1

with open(path1, "w") as f:
    f.writelines(new_lines)

print("clip_dataset.py re-patched ✓")

# ── Verify no syntax errors ───────────────────────────────────────────────────
import ast
with open(path1) as f:
    src = f.read()
try:
    ast.parse(src)
    print("Syntax OK ✓")
except SyntaxError as e:
    print(f"Still broken at line {e.lineno}: {e.msg}")
    print("Context:", src.splitlines()[e.lineno-2:e.lineno+1])

In [ ]:
with open("/content/SMS-Loss/avion/data/clip_dataset.py") as f:
    lines = f.readlines()
for i, l in enumerate(lines[150:175], start=151):
    print(f"{i:3d}| {l}", end="")

In [ ]:
path1 = "/content/SMS-Loss/avion/data/clip_dataset.py"
with open(path1, "r") as f:
    lines = f.readlines()

# Line 159 is index 158 (0-based), the empty elif block is lines 158-159
# We need to insert the body between elif and else
new_lines = []
i = 0
while i < len(lines):
    new_lines.append(lines[i])
    # After the empty `elif 'test' in metadata:` line, insert the body
    if lines[i].rstrip().endswith("elif 'test' in metadata:"):
        # Check next line is `else:` (empty elif body)
        if i + 1 < len(lines) and lines[i+1].rstrip().lstrip().startswith("else:"):
            indent = len(lines[i]) - len(lines[i].lstrip()) + 4  # one extra indent level
            pad = " " * indent
            new_lines.append(f"{pad}import os as _os\n")
            new_lines.append(f"{pad}_rel = getattr(args, 'relevancy_test', None) or getattr(args, 'relevancy_path', None)\n")
            new_lines.append(f"{pad}self.relevancy_mat = pickle.load(open(_rel, 'rb')) if _rel and _os.path.exists(_rel) else None\n")
    i += 1

with open(path1, "w") as f:
    f.writelines(new_lines)

# Verify
import ast
with open(path1) as f:
    src = f.read()
try:
    ast.parse(src)
    print("Syntax OK ✓")
except SyntaxError as e:
    print(f"SyntaxError at line {e.lineno}: {e.msg}")

# Show the fixed region
with open(path1) as f:
    lines = f.readlines()
for i, l in enumerate(lines[153:167], start=154):
    print(f"{i:3d}| {l}", end="")


In [ ]:
HAS_FLASH_ATTN = importlib.util.find_spec("flash_attn") is not None
flash_flag = "--use-flash-attn" if HAS_FLASH_ATTN else ""
print(f"flash_attn: {'enabled' if HAS_FLASH_ATTN else 'disabled'}")

In [ ]:
CKPT = f"{EXP_DIR}/checkpoint_round_1.pt"   # full checkpoint, not _weights.pt

cmd = f"""\
torchrun --nproc_per_node=1 scripts/test_mir.py \\
  --root "{DATA_ROOT}" \\
  --train-metadata "{TRAIN_CSV}" \\
  --val-metadata   "{TEST_CSV}" \\
  --pretrain-model "{CKPT}" \\
  --model CLIP_VITL14 \\
  --project-embed-dim 256 \\
  --use-fast-conv1 \\
  {flash_flag} \\
  --flip \\
  --clip-length 32 \\
  --batch-size 32 \\
  --output-dir "{EXP_DIR}"
"""
print(cmd)
!{cmd}

In [ ]:
!ls drive

In [ ]:
import zipfile, os, pickle, subprocess
import numpy as np

# ── Load your model's output ──────────────────────────────────────────────────
pkl_path = f"{EXP_DIR}/submission.pkl"
with open(pkl_path, "rb") as f:
    raw_out = pickle.load(f)

sim_mat = np.array(raw_out["sim_mat"], dtype=np.float32)
vis_ids = [str(v) for v in raw_out["vis_ids"]]
txt_ids = [str(t) for t in raw_out["txt_ids"]]

print(f"sim_mat : {sim_mat.shape}  dtype={sim_mat.dtype}")
print(f"vis_ids : {len(vis_ids)}  |  txt_ids : {len(txt_ids)}")

# ── SLS scores (adjust to match your model's supervision level) ───────────────
SLS_PT = 2   # pre-training supervision
SLS_TL = 3   # training labels
SLS_TD = 3   # training data

# ── Build a Python-3.7-compatible pickle ──────────────────────────────────────
def make_compat_pickle(sim_mat, vis_ids, txt_ids, sls_pt, sls_tl, sls_td):
    payload = {
        "version":   "0.1",
        "challenge": "multi_instance_retrieval",
        "sls_pt":    sls_pt,
        "sls_tl":    sls_tl,
        "sls_td":    sls_td,
        "sim_mat":   sim_mat,
        "vis_ids":   vis_ids,
        "txt_ids":   txt_ids,
    }
    raw = pickle.dumps(payload, protocol=2)
    # Fix numpy >= 2.0 vs grader's old numpy
    raw = raw.replace(b"numpy._core.multiarray", b"numpy.core.multiarray")
    return raw

# ── Write test.pkl (the grader requires exactly this name) ────────────────────
tmp_pkl = "/tmp/test.pkl"
pkl_bytes = make_compat_pickle(sim_mat, vis_ids, txt_ids, SLS_PT, SLS_TL, SLS_TD)
with open(tmp_pkl, "wb") as f:
    f.write(pkl_bytes)

# Sanity check
check = pickle.loads(pkl_bytes)
assert np.array(check["sim_mat"]).shape == sim_mat.shape, "Shape mismatch after re-load!"
print(f"test.pkl OK — {len(pkl_bytes)/1e6:.1f} MB")

# ── Zip it up ─────────────────────────────────────────────────────────────────
zip_path = f"{EXP_DIR}/submission.zip"
subprocess.run(
    f'cd /tmp && zip -j "{zip_path}" test.pkl',
    shell=True, check=True, capture_output=True
)

print(f"\nReady: {zip_path}  ({os.path.getsize(zip_path)/1e6:.1f} MB)")
print("Submit to: https://www.codabench.org/competitions/12008/")